In [1]:
%pip install pandas pyreadstat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

# --- PHASE 1: Load Data (Bypassing pyreadstat) ---
print("Loading raw files...")
demo = pd.read_sas("data/DEMO_J.xpt", format="xport")
bio  = pd.read_sas("data/BIOPRO_J.xpt", format="xport")
cbc  = pd.read_sas("data/CBC_J.xpt", format="xport")
lux  = pd.read_sas("data/LUX_J.xpt", format="xport")
print("Files loaded successfully.\n")

# --- PHASE 2: Select & Merge ---
# 1. Isolate required markers
demo_s = demo[["SEQN", "RIDAGEYR"]]                    # Age
bio_s  = bio[["SEQN", "LBXSASSI", "LBXSATSI"]]         # AST, ALT
cbc_s  = cbc[["SEQN", "LBXPLTSI"]]                     # Platelets
lux_s  = lux[["SEQN", "LUXSMED"]]                      # Liver stiffness

# 2. Inner Join on Patient ID
merged = demo_s.merge(bio_s, on="SEQN").merge(cbc_s, on="SEQN").merge(lux_s, on="SEQN")

# 3. Save the master raw dataset
merged.to_csv("data/merged_raw.csv", index=False)

print("Merged shape:", merged.shape)
print("\nMissing values (NaNs) in the merged dataset:")
print(merged.isna().sum())

Loading raw files...
Files loaded successfully.

Merged shape: (6401, 6)

Missing values (NaNs) in the merged dataset:
SEQN          0
RIDAGEYR      0
LBXSASSI    519
LBXSATSI    499
LBXPLTSI    343
LUXSMED     451
dtype: int64


In [3]:
clean = merged.dropna(subset=["LUXSMED"]).copy()
print("Shape after dropping missing target:", clean.shape)
print(clean.isna().sum())

Shape after dropping missing target: (5950, 6)
SEQN          0
RIDAGEYR      0
LBXSASSI    421
LBXSATSI    402
LBXPLTSI    252
LUXSMED       0
dtype: int64


In [4]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def impute_predictors(df):
    """MICE-impute AST, ALT, Platelets only — never the target."""
    cols_to_impute = ["LBXSASSI", "LBXSATSI", "LBXPLTSI"]
    imputer = IterativeImputer(random_state=42, max_iter=10)
    df[cols_to_impute] = imputer.fit_transform(df[cols_to_impute])
    print("Remaining NaNs after imputation:\n", df.isna().sum())
    return df

clean = impute_predictors(clean)

Remaining NaNs after imputation:
 SEQN        0
RIDAGEYR    0
LBXSASSI    0
LBXSATSI    0
LBXPLTSI    0
LUXSMED     0
dtype: int64


In [6]:
clean.to_csv("data/clean_imputed.csv", index=False)

In [7]:
import numpy as np

def compute_fib4(df):
    """FIB-4 = (Age × AST) / (Platelets × sqrt(ALT))"""
    df = df.copy()
    df["FIB4"] = (df["RIDAGEYR"] * df["LBXSASSI"]) / (df["LBXPLTSI"] * np.sqrt(df["LBXSATSI"]))
    return df

clean = compute_fib4(clean)
print(clean["FIB4"].describe())

count    5950.000000
mean        0.984666
std         0.898425
min         0.109383
25%         0.443603
50%         0.798856
75%         1.292576
max        37.947332
Name: FIB4, dtype: float64


In [8]:
def isolate_grey_zone(df):
    """Keep only patients in the FIB-4 indeterminate zone."""
    grey_zone = df[(df["FIB4"] >= 1.30) & (df["FIB4"] <= 2.67)].copy()
    print("Grey zone cohort shape:", grey_zone.shape)
    return grey_zone

grey_zone = isolate_grey_zone(clean)

Grey zone cohort shape: (1301, 7)


In [9]:
def define_target(df):
    """1 = advanced fibrosis (stiffness ≥8.0 kPa), 0 = not"""
    df = df.copy()
    df["target"] = (df["LUXSMED"] >= 8.0).astype(int)
    print(df["target"].value_counts())
    return df

grey_zone = define_target(grey_zone)

target
0    1127
1     174
Name: count, dtype: int64


In [10]:
grey_zone.to_csv("data/grey_zone_cohort.csv", index=False)

In [11]:
from sklearn.model_selection import train_test_split

feature_cols = ["RIDAGEYR", "LBXSASSI", "LBXSATSI", "LBXPLTSI"]

X = grey_zone[feature_cols]
y_class = grey_zone["target"]
y_reg = grey_zone["LUXSMED"]

X_train, X_test, y_class_train, y_class_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train positive rate:", y_class_train.mean())
print("Test positive rate:", y_class_test.mean())

Train shape: (1040, 4)
Test shape: (261, 4)
Train positive rate: 0.13365384615384615
Test positive rate: 0.13409961685823754


In [12]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV
from xgboost import XGBClassifier

def train_classifier(X_train, y_train):
    """Train XGBClassifier with class weighting, repeated CV, and grid search tuning."""
    pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    print(f"scale_pos_weight: {pos_weight:.2f}")

    param_grid = {
    "max_depth": [2, 3, 4, 5, 6],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "n_estimators": [50, 100, 200, 300],
    }

    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

    model = XGBClassifier(
        scale_pos_weight=pos_weight,
        eval_metric="logloss",
        random_state=42,
    )

    grid = GridSearchCV(model, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    grid.fit(X_train, y_train)

    print("Best params:", grid.best_params_)
    print(f"Best mean ROC-AUC: {grid.best_score_:.4f}")
    print(f"Std across folds: {grid.cv_results_['std_test_score'][grid.best_index_]:.4f}")

    return grid.best_estimator_

clf_model = train_classifier(X_train, y_class_train)

scale_pos_weight: 6.48
Best params: {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 50}
Best mean ROC-AUC: 0.6069
Std across folds: 0.0573


In [14]:
import pandas as pd

check = X_train.copy()
check["target"] = y_class_train.values
print(check.corr()["target"])

RIDAGEYR   -0.019157
LBXSASSI    0.139814
LBXSATSI    0.193425
LBXPLTSI   -0.029334
target      1.000000
Name: target, dtype: float64


In [16]:
%pip install requests

  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
import pandas as pd

# Load BMX_J (Body Measures - BMI) directly from the local file
bmx = pd.read_sas("data/BMX_J.xpt", format="xport")
bmx_s = bmx[["SEQN", "BMXBMI"]]

# Load BIOPRO_J (albumin, bilirubin, GGT) directly from the local file
bio_full = pd.read_sas("data/BIOPRO_J.xpt", format="xport")
bio_extra = bio_full[["SEQN", "LBXSAL", "LBXSTB", "LBXSGTSI"]]

print("Data loaded successfully.")
print("BMX shape:", bmx_s.shape)
print("Bio extra shape:", bio_extra.shape)

Data loaded successfully.
BMX shape: (8704, 2)
Bio extra shape: (6401, 4)


In [21]:
def add_features(grey_zone, bio_extra, bmx_s):
    """Left-join new biomarkers + derived clinical ratios onto the existing grey-zone cohort."""
    df = grey_zone.merge(bio_extra, on="SEQN", how="left").merge(bmx_s, on="SEQN", how="left")
    df["AST_ALT_ratio"] = df["LBXSASSI"] / df["LBXSATSI"]
    df["AST_Platelet_ratio"] = df["LBXSASSI"] / df["LBXPLTSI"]
    print("Shape:", df.shape)
    print("New NaNs:\n", df[["LBXSAL","LBXSTB","LBXSGTSI","BMXBMI"]].isna().sum())
    return df

grey_zone_v2 = add_features(grey_zone, bio_extra, bmx_s)

Shape: (1301, 14)
New NaNs:
 LBXSAL      61
LBXSTB      62
LBXSGTSI    62
BMXBMI      15
dtype: int64


In [22]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

def impute_new_features(df):
    cols = ["LBXSAL", "LBXSTB", "LBXSGTSI", "BMXBMI"]
    imputer = IterativeImputer(random_state=42, max_iter=10)
    df[cols] = imputer.fit_transform(df[cols])
    print("Remaining NaNs:\n", df[cols].isna().sum())
    return df

grey_zone_v2 = impute_new_features(grey_zone_v2)

Remaining NaNs:
 LBXSAL      0
LBXSTB      0
LBXSGTSI    0
BMXBMI      0
dtype: int64


In [23]:
from sklearn.model_selection import train_test_split

feature_cols_v2 = ["RIDAGEYR", "LBXSASSI", "LBXSATSI", "LBXPLTSI",
                    "LBXSAL", "LBXSTB", "LBXSGTSI", "BMXBMI",
                    "AST_ALT_ratio", "AST_Platelet_ratio"]

X_v2 = grey_zone_v2[feature_cols_v2]
y_v2 = grey_zone_v2["target"]

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2, y_v2, test_size=0.2, random_state=42, stratify=y_v2
)

In [24]:
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV
from xgboost import XGBClassifier

def train_classifier_v2(X_train, y_train):
    pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    param_grid = {"max_depth": [2,3,4,5,6], "learning_rate": [0.01,0.05,0.1,0.2], "n_estimators": [50,100,200,300]}
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
    model = XGBClassifier(scale_pos_weight=pos_weight, eval_metric="logloss", random_state=42)
    grid = GridSearchCV(model, param_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    grid.fit(X_train, y_train)
    print("XGBoost best params:", grid.best_params_)
    print(f"XGBoost mean ROC-AUC: {grid.best_score_:.4f} (+/- {grid.cv_results_['std_test_score'][grid.best_index_]:.4f})")
    return grid.best_estimator_

xgb_model_v2 = train_classifier_v2(X_train_v2, y_train_v2)

XGBoost best params: {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 200}
XGBoost mean ROC-AUC: 0.7361 (+/- 0.0515)


In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

def train_logreg(X_train, y_train):
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))])
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
    scores = cross_val_score(pipe, X_train, y_train, scoring="roc_auc", cv=cv)
    print(f"Logistic Regression mean ROC-AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")
    return pipe.fit(X_train, y_train)

logreg_model = train_logreg(X_train_v2, y_train_v2)

Logistic Regression mean ROC-AUC: 0.7358 (+/- 0.0619)


In [26]:
from xgboost import XGBRegressor
from sklearn.model_selection import RepeatedKFold

def train_regressor(X_train, y_train):
    """Predict exact kPa value using the same expanded feature set."""
    param_grid = {"max_depth": [2,3,4,5,6], "learning_rate": [0.01,0.05,0.1,0.2], "n_estimators": [50,100,200,300]}
    cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)
    model = XGBRegressor(random_state=42)
    grid = GridSearchCV(model, param_grid, scoring="neg_mean_absolute_error", cv=cv, n_jobs=-1)
    grid.fit(X_train, y_train)
    print("Regressor best params:", grid.best_params_)
    print(f"Mean MAE: {-grid.best_score_:.4f} (+/- {grid.cv_results_['std_test_score'][grid.best_index_]:.4f}) kPa")
    return grid.best_estimator_

y_reg_train_v2 = grey_zone_v2.loc[X_train_v2.index, "LUXSMED"]
reg_model_v2 = train_regressor(X_train_v2, y_reg_train_v2)

Regressor best params: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 50}
Mean MAE: 2.1859 (+/- 0.2429) kPa


In [27]:
import joblib

joblib.dump(xgb_model_v2, "results/xgb_classifier.pkl")
joblib.dump(reg_model_v2, "results/xgb_regressor.pkl")
joblib.dump(logreg_model, "results/logreg_comparison.pkl")

# Save test set too — untouched until Phase 7
X_test_v2.to_csv("data/X_test_v2.csv", index=False)
y_test_v2.to_csv("data/y_test_v2.csv", index=False)